# Financial Ratios and Portfolio Analysis
In this notebook, we will be calculating several financial ratios for Lending Club's loans and doing an analysis of their key portfolio metrics.

In [0]:
spark.conf.set(
    "fs.azure.account.key.lcdevadls.blob.core.windows.net",
    "REDACTED"
)

spark.conf.set(
    "fs.azure.account.key.lcstgadls.blob.core.windows.net",
    "REDACTED"
)

spark.conf.set(
    "fs.azure.account.key.lcprodadls.blob.core.windows.net",
    "REDACTED"
)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, sum, lit
import pyspark.sql.functions as F
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

INPUT_PATH = "wasbs://lcloansdev@lcdevadls.blob.core.windows.net/"

df = spark.read.parquet(INPUT_PATH)

# Caching this as its our main dataframe that we will be using for analysis
df.cache()

print(f"Total rows: {df.count():,}")
df.show(5)

#### Helper Columns and Casting
First, we are going to create any helper columns and caste existing columns to numeric types just in case they aren't already.

In [0]:
cols_to_float = [
    "tot_cur_bal", "tot_hi_cred_lim", "revol_bal",
    "annual_inc", "bc_util", "bc_open_to_buy",
    "open_acc", "total_acc", "num_accts_ever_120_pd",
    "annual_inc_joint", "dti", "dti_joint"
]

for col in cols_to_float:
    df = df.withColumn(col, F.col(col).cast("float"))

In [0]:
from pyspark.sql.functions import when, col, sum, count

# For default flag
df = df.withColumn(
    "default_flag",
    when(col("loan_status").isin("Charged Off", "Default", "Late (31-120 days)"), 1).otherwise(0)
)

# Let's also check the overall default rate
df.agg(
    count("*").alias("total_loans"),
    sum("default_flag").alias("total_defaults"),
).withColumn(
    "default_rate",
    when(col("total_loans") != 0, col("total_defaults") / col("total_loans")).otherwise(None)
).show()

#### Credit Utilization Ratios
This analysis examines how credit utilization patterns vary across Lending Club loan grades. We compute three key ratios:
- Current Balance Ratio – total balances relative to total high-limit credit.
- Revolving Balance to Income – revolving debt relative to annual income.
- Credit Card Utilization – fraction of open credit card limits being used.

Understanding these ratios helps us profile the borrower's financial health, model credit risk and complements the default analysis from the loan_performance_analysis notebook. It can inform portfolio allocation decisions and risk-based pricing.


In [0]:
# Filtering out rows with zero or null denominators to avoid division errors while calculating ratios
df = df.filter(
    (F.col("tot_hi_cred_lim").isNotNull()) & (F.col("tot_hi_cred_lim") != 0) &
    (F.col("annual_inc").isNotNull()) & (F.col("annual_inc") != 0) &
    (F.col("bc_open_to_buy").isNotNull()) & (F.col("bc_open_to_buy") != 0) &
    (F.col("total_acc").isNotNull()) & (F.col("total_acc") != 0) &
    (F.col("annual_inc_joint").isNotNull()) & (F.col("annual_inc_joint") != 0)
)

df = df.withColumn("cur_bal_ratio", F.col("tot_cur_bal") / F.col("tot_hi_cred_lim")) \
       .withColumn("revol_income_ratio", F.col("revol_bal") / F.col("annual_inc")) \
       .withColumn("bc_util_ratio", F.col("bc_util") / F.col("bc_open_to_buy"))

revol_income_99 = df.approxQuantile("revol_income_ratio", [0.99], 0.0)[0]

# Cap the revol_income_ratio at the 99th percentile to reduce outlier impact because this particular ratio has some extreme outliers
df = df.withColumn(
    "revol_income_ratio",
    when(col("revol_income_ratio") > revol_income_99, revol_income_99).otherwise(col("revol_income_ratio"))
)

# Aggregation by grade
grade_ratios = df.groupBy("grade").agg(
    F.round(F.avg("cur_bal_ratio"), 4).alias("avg_cur_bal_ratio"),
    F.round(F.avg("revol_income_ratio"), 4).alias("avg_revol_income_ratio"),
    F.round(F.avg("bc_util_ratio"), 4).alias("avg_bc_util_ratio")
).orderBy("grade")

grade_ratios.cache()

print("Grade-Level Credit Utilization Ratios:")
grade_ratios.show()


In [0]:
pdf = grade_ratios.toPandas()

# This format will make it easier to plot multiple ratios in a single plot
pdf = pdf.melt(
    id_vars="grade",
    value_vars=["avg_cur_bal_ratio", "avg_revol_income_ratio", "avg_bc_util_ratio"],
    var_name="ratio_type",
    value_name="value"
)

# Renaming these columns for better readability in the plot
pdf["ratio_type"] = pdf["ratio_type"].map({
    "avg_cur_bal_ratio": "Current Balance / Credit Limit",
    "avg_revol_income_ratio": "Revolving / Income",
    "avg_bc_util_ratio": "BC Utilization"
})

plt.figure(figsize=(8,4))
sns.barplot(data=pdf, x="grade", y="value", hue="ratio_type", palette="mako")

plt.title("Average Credit Utilization Ratios by Loan Grade")
plt.ylabel("Ratio")
plt.xlabel("Loan Grade")
plt.legend(title="Ratio Type")
plt.tight_layout()
plt.show()

As expected, lower-grade (riskier) borrowers generally have higher credit utilization ratios, indicating they are financially more stretched. They also use a larger fraction of their credit cards. We see a similar trend for the revolving debt to income ratio too but it isn't as drastic.

#### Account Activity and Delinquency
Account activity and delinquency are important indicators of borrower behavior beyond just credit scores or loan performance.
- active_ratio measures how many of a borrower’s total accounts are currently active. A higher ratio can indicate a borrower is actively using credit.
- never_delinquent captures whether the borrower has ever been seriously delinquent (120+ days). This indicates reliability and repayment discipline.

By aggregating these metrics at the grade level, we can see whether borrowers in lower vs higher credit grades differ in how they use credit and manage repayments.

In [0]:
df = df.withColumn("active_ratio", F.col("open_acc") / F.col("total_acc")) \
       .withColumn("never_delinquent", (F.col("num_accts_ever_120_pd") == 0).cast("double"))

grade_activity = df.groupBy("grade").agg(
    F.round(F.avg("active_ratio"), 4).alias("avg_active_ratio"),
    F.round(F.avg("never_delinquent"), 4).alias("pct_never_delinquent")
).orderBy("grade")

grade_activity.cache()

print("Grade-Level Account Activity & Delinquency:")
grade_activity.show()


In [0]:
pdf = grade_activity.toPandas()

pdf = pdf.melt(
    id_vars="grade",
    value_vars=["avg_active_ratio", "pct_never_delinquent"],
    var_name="metric",
    value_name="value"
)

pdf["metric"] = pdf["metric"].map({
    "avg_active_ratio": "Active Accounts / Total Accounts",
    "pct_never_delinquent": "% Never Delinquent"
})

plt.figure(figsize=(8,4))
sns.barplot(data=pdf, x="grade", y="value", hue="metric", palette="mako")

plt.title("Average Account Activity and Delinquency by Loan Grade")
plt.ylabel("Value / Percentage")
plt.xlabel("Loan Grade")
plt.legend(title="Metric")
plt.tight_layout()
plt.show()

Active Ratio gradually increases from A to G, suggesting that lower-grade borrowers maintain more active accounts. This might indicate higher credit usage or reliance on multiple accounts. The Never Delinquent score enerally declines from A to F, meaning lower-grade borrowers have historically experienced more serious delinquencies. Grade G is a slight anomaly in this trend but that is likely due to fewer loans or special cases in the dataset.

#### Joint vs Individual Loan Comparisons

In [0]:
df = df.withColumn("income_ratio_joint", F.col("annual_inc") / F.col("annual_inc_joint")) \
       .withColumn("dti_diff", F.col("dti") - F.col("dti_joint"))

grade_joint = df.groupBy("grade").agg(
    F.round(F.avg("income_ratio_joint"), 4).alias("avg_income_ratio_joint"),
    F.round(F.avg("dti_diff"), 4).alias("avg_dti_diff")
).orderBy("grade")

grade_joint.cache()

print("Grade-Level Joint vs Individual Loan Comparisons:")
grade_joint.show()


The average income ratio is roughly stable around 0.48–0.51, meaning most borrowers contribute roughly half of household income, across grades. No strong trend with credit grade.

#### State-level Summary
Creditworthiness and debt behavior can vary geographically due to differences in cost of living and local lending practices. Aggregating borrower-level metrics at the state level allows us to identify regions where borrowers are more financially stretched or more conservative with credit usage.

In [0]:
state_summary = df.groupBy("addr_state").agg(
    F.round(F.avg("dti"), 4).alias("avg_dti"),
    F.round(F.avg("revol_util"), 4).alias("avg_revol_util"),
    F.round(F.avg("cur_bal_ratio"), 4).alias("avg_cur_bal_ratio"),
    F.count("*").alias("num_loans")
).orderBy(F.desc("avg_dti"))

print("State-Level Summary:")
state_summary.show(50)


- UT, NM, VT, MT show the highest average DTI (35–55%), indicating borrowers in these states carry more debt relative to income.
- This may reflect local lending practices, housing costs, or borrower behavior.
- WY and HI have particularly high revolving utilization (50–55%), suggesting borrowers rely more heavily on credit cards.
- WY and HI have particularly high revolving utilization (50–55%), suggesting borrowers rely more heavily on credit cards.
- States like WY, MT, and CO have higher avg_cur_bal_ratio (>0.75), meaning borrowers are using a larger fraction of their high-limit credit.
- States with lower ratios (NY, NJ, AK) show more conservative credit usage.
- CA, TX, and FL dominate the sample, which means national-level averages are heavily influenced by these large states.
- Smaller states like DC or AK may show extreme values due to low sample sizes.

#### Portfolio Analysis
This section focuses on portfolio-level performance metrics: cash flows, ROI, and default rates aggregated by different loan/borrower segments (grade, term, state).

In [0]:
df_inv = df.withColumn(
    "service_fee",
    F.coalesce(F.col("collection_recovery_fee"), F.lit(0.0))
).withColumn(
    "net_cash",
    F.col("total_pymnt") + F.coalesce(F.col("recoveries"), F.lit(0.0)) -
    F.col("funded_amnt") - F.col("service_fee")
)


In [0]:
df_inv = df_inv.withColumn(
    "end_date",
    F.coalesce(F.col("last_pymnt_d"), F.current_date())
).withColumn(
    "years_held",
    F.when(F.col("issue_d").isNotNull(),
           F.months_between(F.col("end_date"), F.col("issue_d")) / 12.0)
     .otherwise(F.lit(None).cast("double"))
).withColumn(
    "years_held_safe",
    F.when(F.col("years_held") < 0.1, 0.1).otherwise(F.col("years_held"))
).withColumn(
    "roi_annualized",
    F.when(F.col("funded_amnt") != 0,
           (F.col("net_cash") / F.col("funded_amnt")) / F.col("years_held_safe"))
     .otherwise(F.lit(None))
)

Since very new loans can have extreme annualized ROI due to short holding periods, we filter to loans that have paid at least 50% of principal back


In [0]:
df_inv = df_inv.filter(
    (F.col("total_pymnt") >= 0.5 * F.col("funded_amnt")) &  # at least 50% paid
    (F.col("funded_amnt") > 0)
)

#caching this dataframe as we will be using it for multiple analyses and dont want to recompute each time
df_inv.cache() 

Grade level analysis

In [0]:
portfolio_grade = df_inv.groupBy("grade").agg(
    F.count("*").alias("n_loans"),
    F.sum("funded_amnt").alias("total_principal"),
    F.sum("net_cash").alias("total_net_cash"),
    F.round(F.avg("roi_annualized"), 4).alias("avg_roi_annualized"),
    F.round(F.avg(F.col("default_flag").cast("double")), 4).alias("realized_default_rate")
).orderBy("grade")

portfolio_grade.show()


- ROI generally increases with grade risk: grades D–F have higher average ROI than A–C, reflecting higher interest rates on riskier loans.
- Total principal is concentrated in higher grades (A through D), meaning investors tend to invest more in moderate-risk loans.
- Default rates are all showing 0 — this is because we filtered out loans that haven’t repaid 50% of principal. Essentially, many loans that would have defaulted might are being excluded by this filter. However, ROI was the priority in this particular analysis so we still filter regardless for a more accurate ROI picture.

Portfolio-level aggregation by grade and term

In [0]:
portfolio_grade_term = df_inv.groupBy("grade", "term").agg(
    F.count("*").alias("n_loans"),
    F.sum("funded_amnt").alias("total_principal"),
    F.sum("net_cash").alias("total_net_cash"),
    F.round(F.avg("roi_annualized"), 4).alias("avg_roi_annualized"),
    F.round(F.avg(F.col("default_flag").cast("double")), 4).alias("realized_default_rate")
).orderBy("grade", "term")

portfolio_grade_term.show()

- ROI varies by term within grades: e.g. Grade C loans show negative ROI for 36-month loans but positive ROI for 60-month loans. This suggests that longer-term loans may provide more opportunity to accumulate interest, or shorter-term loans may be partially repaid but not enough to cover principal.

- Default rates: Mostly 0 due to filtering for loans with significant repayment or because many loans are still in progress. Grade F 60-month loans show a non-zero default rate (0.111), indicating some defaults in longer-term riskier loans.

- Net cash differences: Negative net cash for some short-term loans in high grades indicates partial repayments or losses.

Portfolio-level aggregation by state

In [0]:
portfolio_state = df_inv.groupBy("addr_state").agg(
    F.count("*").alias("n_loans"),
    F.sum("funded_amnt").alias("total_principal"),
    F.sum("net_cash").alias("total_net_cash"),
    F.round(F.avg("roi_annualized"), 4).alias("avg_roi_annualized"),
    F.round(F.avg(F.col("default_flag").cast("double")), 4).alias("realized_default_rate")
).orderBy(F.desc("avg_roi_annualized"))

portfolio_state.show(100)

- Highest ROI observed in small-volume states like DC, HI, and WV. These states have very few loans, so ROI is influenced by individual loan outcomes (small sample size bias).
- Larger states with more loans (CA, TX, FL, NY) have modest positive ROI, but some negative ROI is observed in NY, GA, UT, AL, KY, SC, MD, LA — likely reflecting partial repayments or losses in certain loans.